In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve
)

# =========================================================
# 1. LOAD DATA
# =========================================================
# Change path if needed
df = pd.read_csv("../data/disease10k.csv")

# Target column
target_col = "has_disease"

# Basic check
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df[target_col].value_counts())
print("\nClass percentages:")
print(df[target_col].value_counts(normalize=True).round(4))

# =========================================================
# 2. SPLIT FEATURES / TARGET
# =========================================================
X = df.drop(columns=[target_col])
y = df[target_col]

# =========================================================
# 3. TRAIN / VALIDATION SPLIT
#    90% train, 10% validation
# =========================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive ratio:", y_train.mean().round(4))
print("y_val positive ratio:", y_val.mean().round(4))

# =========================================================
# 4. IDENTIFY NUMERIC / CATEGORICAL COLUMNS
# =========================================================
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nNumeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

# =========================================================
# 5. PREPROCESSING
# =========================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =========================================================
# 6. MODEL PIPELINE
# =========================================================
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        solver="liblinear",
        random_state=42
    ))
])

# =========================================================
# 7. GRID SEARCH WITH 10-FOLD CV
#    Optimize for recall
# =========================================================
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 50, 100],
    "model__penalty": ["l1", "l2"]
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(X_train, y_train)

print("\n==============================")
print("BEST GRID SEARCH RESULTS")
print("==============================")
print("Best params:", grid_search.best_params_)
print("Best CV recall:", round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_

# =========================================================
# 8. VALIDATION PROBABILITIES
# =========================================================
y_val_proba = best_model.predict_proba(X_val)[:, 1]

# Default threshold = 0.5
y_val_pred_default = (y_val_proba >= 0.5).astype(int)

print("\n==============================")
print("VALIDATION RESULTS @ THRESHOLD = 0.5")
print("==============================")
print("Recall:", round(recall_score(y_val, y_val_pred_default, zero_division=0), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_default, zero_division=0), 4))
print("F1:", round(f1_score(y_val, y_val_pred_default, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_default))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_default, zero_division=0))

# =========================================================
# 9. THRESHOLD TUNING ON VALIDATION SET
#    Goal: recall >= 0.90
#    Then maximize precision
# =========================================================
target_recall = 0.90
thresholds = np.arange(0.01, 1.00, 0.01)

threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    recall = recall_score(y_val, y_val_pred, zero_division=0)
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    threshold_results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

# Keep only thresholds meeting recall constraint
valid_thresholds = threshold_df[threshold_df["recall"] >= target_recall].copy()

print("\n==============================")
print("THRESHOLD SEARCH")
print("==============================")
print("Thresholds meeting recall >= 0.90:", len(valid_thresholds))

if len(valid_thresholds) > 0:
    # Choose highest precision among them
    # If tie, choose higher threshold
    best_threshold_row = valid_thresholds.sort_values(
        by=["precision", "threshold"],
        ascending=[False, False]
    ).iloc[0]

    best_threshold = best_threshold_row["threshold"]

    print("\nBest threshold under recall constraint:")
    print(best_threshold_row)

    y_val_pred_best = (y_val_proba >= best_threshold).astype(int)

    print("\n==============================")
    print(f"VALIDATION RESULTS @ THRESHOLD = {best_threshold:.2f}")
    print("==============================")
    print("Recall:", round(recall_score(y_val, y_val_pred_best, zero_division=0), 4))
    print("Precision:", round(precision_score(y_val, y_val_pred_best, zero_division=0), 4))
    print("F1:", round(f1_score(y_val, y_val_pred_best, zero_division=0), 4))
    print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba), 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, y_val_pred_best))

    print("\nClassification Report:")
    print(classification_report(y_val, y_val_pred_best, zero_division=0))

    print("\nTop threshold candidates:")
    print(
        valid_thresholds.sort_values(
            by=["precision", "threshold"],
            ascending=[False, False]
        ).head(10)
    )

else:
    print("\nNo threshold achieved recall >= 0.90 on validation.")
    print("You may need to:")
    print("- try another model")
    print("- expand the grid")
    print("- use resampling")
    print("- engineer better features")

# =========================================================
# 10. OPTIONAL: SEE ALL THRESHOLD RESULTS
# =========================================================
print("\nAll threshold results head:")
print(threshold_df.head())

print("\nAll threshold results tail:")
print(threshold_df.tail())

Dataset shape: (10000, 7)

Class distribution:
has_disease
0    9851
1     149
Name: count, dtype: int64

Class percentages:
has_disease
0    0.9851
1    0.0149
Name: proportion, dtype: float64

Shapes:
X_train: (9000, 6)
X_val: (1000, 6)
y_train positive ratio: 0.0149
y_val positive ratio: 0.015

Numeric columns: ['age', 'bmi', 'blood_pressure', 'glucose_level', 'family_history']
Categorical columns: ['physical_activity_level']
Fitting 10 folds for each of 12 candidates, totalling 120 fits

BEST GRID SEARCH RESULTS
Best params: {'model__C': 0.01, 'model__penalty': 'l1'}
Best CV recall: 0.8286

VALIDATION RESULTS @ THRESHOLD = 0.5
Recall: 0.8667
Precision: 0.0602
F1: 0.1126
ROC-AUC: 0.9291

Confusion Matrix:
[[782 203]
 [  2  13]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.79      0.88       985
           1       0.06      0.87      0.11        15

    accuracy                           0.80      1000
   macro avg     

In [3]:
valid_thresholds.sort_values(
    by=["precision", "threshold"],
    ascending=[False, False]
).head(10)

,threshold,recall,precision,f1
38,0.39,0.933333,0.045752,0.087227
37,0.38,0.933333,0.044586,0.085106
36,0.37,0.933333,0.043614,0.083333
35,0.36,0.933333,0.042042,0.080460
34,0.35,0.933333,0.040816,0.078212
33,0.34,0.933333,0.040346,0.077348
32,0.33,0.933333,0.039548,0.075881
31,0.32,0.933333,0.038462,0.073879
30,0.31,0.933333,0.037534,0.072165
27,0.28,1.000000,0.036855,0.071090


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline  # IMPORTANT
from imblearn.over_sampling import SMOTE

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("../data/disease10k.csv")
target_col = "has_disease"

X = df.drop(columns=[target_col])
y = df[target_col]

# =========================================================
# 2. TRAIN / VALIDATION SPLIT
# =========================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

# =========================================================
# 3. COLUMN TYPES
# =========================================================
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# =========================================================
# 4. PREPROCESSING
# =========================================================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# =========================================================
# 5. PIPELINE WITH SMOTE
# =========================================================
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        random_state=42
    ))
])

# =========================================================
# 6. GRID SEARCH (10-FOLD CV)
# =========================================================
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 50],
    "model__penalty": ["l1", "l2"]
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest params:", grid.best_params_)
print("Best CV recall:", grid.best_score_)

best_model = grid.best_estimator_

# =========================================================
# 7. VALIDATION PROBABILITIES
# =========================================================
y_val_proba = best_model.predict_proba(X_val)[:, 1]

# =========================================================
# 8. THRESHOLD TUNING (RECALL >= 0.90)
# =========================================================
thresholds = np.arange(0.01, 1.00, 0.01)

results = []

for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    
    recall = recall_score(y_val, y_pred, zero_division=0)
    precision = precision_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    results.append((t, recall, precision, f1))

results_df = pd.DataFrame(results, columns=["threshold", "recall", "precision", "f1"])

valid = results_df[results_df["recall"] >= 0.90]

print("\nThresholds meeting recall >= 0.90:", len(valid))

if len(valid) > 0:
    best_row = valid.sort_values(
        by=["precision", "threshold"],
        ascending=[False, False]
    ).iloc[0]

    print("\nBEST THRESHOLD:")
    print(best_row)

    best_t = best_row["threshold"]
    y_best = (y_val_proba >= best_t).astype(int)

    print("\nFINAL VALIDATION METRICS:")
    print("Recall:", recall_score(y_val, y_best))
    print("Precision:", precision_score(y_val, y_best))
    print("F1:", f1_score(y_val, y_best))

else:
    print("No threshold meets recall >= 0.90")

Fitting 10 folds for each of 10 candidates, totalling 100 fits

Best params: {'model__C': 0.1, 'model__penalty': 'l2'}
Best CV recall: 0.7923076923076924

Thresholds meeting recall >= 0.90: 31

BEST THRESHOLD:
threshold    0.310000
recall       0.933333
precision    0.047458
f1           0.090323
Name: 30, dtype: float64

FINAL VALIDATION METRICS:
Recall: 0.9333333333333333
Precision: 0.04745762711864407
F1: 0.09032258064516129
